In [6]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
import time

target_title = "Awards/degrees conferred by program (6-digit CIP code), award level"

base_url = "https://nces.ed.gov/ipeds/datacenter/DataFiles.aspx"

headers = {
    "User-Agent": "Mozilla/5.0"
}

results = []

for year in range(1980, 2025):
    params = {
        "year": year,
        "sid": "31f7cf45-d241-4af3-9028-3db7dd854b71",
        "rtid": 7
    }

    try:
        response = requests.get(
            base_url,
            params=params,
            headers=headers,
            timeout=20,
            allow_redirects=True
        )

        soup = BeautifulSoup(response.text, "html.parser")

        page_text = soup.get_text(" ", strip=True)

        # Search exact title in the whole page text
        if target_title.lower() in page_text.lower():
            found_title = target_title
            status = "available"
        else:
            found_title = "not available"
            status = "not available"

        results.append({
            "year": year,
            "searched_title": target_title,
            "found_title": found_title,
            "status": status,
            "url": response.url
        })

        print(year, status)

        time.sleep(0.5)

    except Exception as e:
        results.append({
            "year": year,
            "searched_title": target_title,
            "found_title": "not available",
            "status": f"error: {e}",
            "url": ""
        })

# Create table
df = pd.DataFrame(results)

# Show full table
print(df)

# Save to CSV
#df.to_csv("ipeds_awards_degrees_1984_2024_search.csv", index=False)

#df

1980 not available
1981 not available
1982 not available
1983 not available
1984 available
1985 available
1986 available
1987 available
1988 available
1989 available
1990 available
1991 available
1992 available
1993 available
1994 available
1995 available
1996 available
1997 available
1998 available
1999 available
2000 available
2001 available
2002 available
2003 available
2004 available
2005 available
2006 available
2007 available
2008 available
2009 available
2010 available
2011 available
2012 available
2013 available
2014 available
2015 available
2016 available
2017 available
2018 available
2019 available
2020 available
2021 available
2022 available
2023 available
2024 available
    year                                     searched_title  \
0   1980  Awards/degrees conferred by program (6-digit C...   
1   1981  Awards/degrees conferred by program (6-digit C...   
2   1982  Awards/degrees conferred by program (6-digit C...   
3   1983  Awards/degrees conferred by program (6-digit C.

| File             | Meaning                                    | Use it?                      |
| ---------------- | ------------------------------------------ | ---------------------------- |
| `c2010_a.csv`    | Original/provisional 2010 Completions file | Usually skip if `_rv` exists |
| `c2010_a_rv.csv` | Revised/final 2010 Completions file        | **Use this one**             |


In [23]:
import requests
from bs4 import BeautifulSoup
from pathlib import Path
from urllib.parse import urljoin
from io import BytesIO
import zipfile
import shutil
import time

# ==================================================
# SETTINGS
# ==================================================

target_title = "Awards/degrees conferred by program (6-digit CIP code), award level"

start_year = 1984
end_year = 2024

folder = Path("degree_year")

# True = delete old degree_year folder first
# This removes old wrong files before starting fresh
CLEAN_FOLDER_FIRST = True

base_url = "https://nces.ed.gov/ipeds/datacenter/DataFiles.aspx"

session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0"
})


# ==================================================
# CREATE CLEAN FOLDER
# ==================================================

if CLEAN_FOLDER_FIRST and folder.exists():
    shutil.rmtree(folder)

folder.mkdir(exist_ok=True)


# ==================================================
# HELPER FUNCTIONS
# ==================================================

def clean_text(text):
    return " ".join(text.lower().split())


def get_year_page(year):
    params = {
        "year": year,
        "rtid": 7
    }

    response = session.get(base_url, params=params, timeout=30)
    response.raise_for_status()

    return response.url, response.text


def find_data_file_link(html, page_url):
    soup = BeautifulSoup(html, "html.parser")
    target_clean = clean_text(target_title)

    rows = soup.find_all("tr")

    for row in rows:
        row_text = clean_text(row.get_text(" ", strip=True))

        if target_clean not in row_text:
            continue

        links = row.find_all("a", href=True)

        for link in links:
            href = link["href"]
            text = link.get_text(" ", strip=True)

            check = (href + " " + text).lower()

            # Skip non-data files
            bad_words = [
                "stata",
                "spss",
                "sas",
                "dict",
                "dictionary",
                "program"
            ]

            if any(bad in check for bad in bad_words):
                continue

            if ".zip" in check:
                return urljoin(page_url, href)

    return None


def choose_best_csv_info(zip_file):
    csv_infos = [
        info for info in zip_file.infolist()
        if info.filename.lower().endswith(".csv")
    ]

    if not csv_infos:
        return None

    rv_infos = [
        info for info in csv_infos
        if Path(info.filename).name.lower().endswith("_rv.csv")
    ]

    # Use revised file first if available
    if rv_infos:
        return max(rv_infos, key=lambda x: x.file_size)

    # Otherwise use normal CSV
    return max(csv_infos, key=lambda x: x.file_size)


def save_csv_year_filename_only(zip_file, csv_info, year):
    original_filename = Path(csv_info.filename).name.lower()

    # Example:
    # 2010_c2010_a_rv.csv
    output_filename = f"{year}_{original_filename}"
    output_path = folder / output_filename

    # Copy CSV directly.
    # Do NOT change inside the CSV.
    with zip_file.open(csv_info) as source, open(output_path, "wb") as output:
        shutil.copyfileobj(source, output)

    return output_filename


def download_and_save_one_year(year, data_file_url):
    zip_response = session.get(data_file_url, timeout=90)
    zip_response.raise_for_status()

    with zipfile.ZipFile(BytesIO(zip_response.content), "r") as zip_file:
        csv_info = choose_best_csv_info(zip_file)

        if csv_info is None:
            return None, "no csv found"

        saved_filename = save_csv_year_filename_only(
            zip_file=zip_file,
            csv_info=csv_info,
            year=year
        )

        if Path(csv_info.filename).name.lower().endswith("_rv.csv"):
            file_type = "revised"
        else:
            file_type = "normal"

        return saved_filename, file_type


# ==================================================
# MAIN AUTO DOWNLOAD
# ==================================================

total_start = time.perf_counter()

print("Auto download started")
print(f"Saving files into folder: {folder}")
print("-" * 60)

available_count = 0
not_available_count = 0
error_count = 0

for year in range(start_year, end_year + 1):
    year_start = time.perf_counter()

    print(f"Checking {year}...")

    try:
        page_url, html = get_year_page(year)

        data_file_url = find_data_file_link(html, page_url)

        if data_file_url is None:
            not_available_count += 1
            print("  Not available")
            print()
            continue

        saved_filename, file_type = download_and_save_one_year(
            year,
            data_file_url
        )

        year_seconds = round(time.perf_counter() - year_start, 2)
        total_so_far = round(time.perf_counter() - total_start, 2)

        if saved_filename is None:
            error_count += 1
            print(f"  Error: {file_type}")
            print()
            continue

        available_count += 1

        print(f"  Saved: {saved_filename}")
        print(f"  File type used: {file_type}")
        print(f"  Year time: {year_seconds} seconds")
        print(f"  Total time so far: {total_so_far} seconds")
        print()

    except Exception as e:
        error_count += 1

        year_seconds = round(time.perf_counter() - year_start, 2)

        print(f"  Error: {e}")
        print(f"  Year time: {year_seconds} seconds")
        print()


# ==================================================
# FINAL TIME
# ==================================================

total_seconds = round(time.perf_counter() - total_start, 2)
total_minutes = round(total_seconds / 60, 2)

print("-" * 60)
print("Auto download finished")
print(f"Total time: {total_seconds} seconds")
print(f"Total time: {total_minutes} minutes")
print(f"Available files saved: {available_count}")
print(f"Not available years: {not_available_count}")
print(f"Errors: {error_count}")
print(f"Folder: {folder}")

Auto download started
Saving files into folder: degree_year
------------------------------------------------------------
Checking 1984...
  Saved: 1984_c1984_cip.csv
  File type used: normal
  Year time: 3.89 seconds
  Total time so far: 3.89 seconds

Checking 1985...
  Saved: 1985_c1985_cip.csv
  File type used: normal
  Year time: 1.27 seconds
  Total time so far: 5.15 seconds

Checking 1986...
  Saved: 1986_c1986_cip.csv
  File type used: normal
  Year time: 1.43 seconds
  Total time so far: 6.59 seconds

Checking 1987...
  Saved: 1987_c1987_cip.csv
  File type used: normal
  Year time: 4.09 seconds
  Total time so far: 10.67 seconds

Checking 1988...
  Saved: 1988_c1988_cip.csv
  File type used: normal
  Year time: 3.58 seconds
  Total time so far: 14.25 seconds

Checking 1989...
  Saved: 1989_c1989_cip.csv
  File type used: normal
  Year time: 3.56 seconds
  Total time so far: 17.81 seconds

Checking 1990...
  Saved: 1990_c8990cip.csv
  File type used: normal
  Year time: 2.88 sec

# Error correct

In [25]:
###########Check why error###########
import requests
from bs4 import BeautifulSoup
from pathlib import Path
from urllib.parse import urljoin
from io import BytesIO
import zipfile
import shutil
import time

# ==================================================
# SETTINGS
# ==================================================

target_title = "Awards/degrees conferred by program (6-digit CIP code), award level"

start_year = 2023
end_year = 2024

folder = Path("CorrectCheckYear")

# True = delete old degree_year folder first
# This removes old wrong files before starting fresh
CLEAN_FOLDER_FIRST = True

base_url = "https://nces.ed.gov/ipeds/datacenter/DataFiles.aspx"

session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0"
})


# ==================================================
# CREATE CLEAN FOLDER
# ==================================================

if CLEAN_FOLDER_FIRST and folder.exists():
    shutil.rmtree(folder)

folder.mkdir(exist_ok=True)


# ==================================================
# HELPER FUNCTIONS
# ==================================================

def clean_text(text):
    return " ".join(text.lower().split())


def get_year_page(year):
    params = {
        "year": year,
        "rtid": 7
    }

    response = session.get(base_url, params=params, timeout=30)
    response.raise_for_status()

    return response.url, response.text


def find_data_file_link(html, page_url):
    soup = BeautifulSoup(html, "html.parser")
    target_clean = clean_text(target_title)

    rows = soup.find_all("tr")

    for row in rows:
        row_text = clean_text(row.get_text(" ", strip=True))

        if target_clean not in row_text:
            continue

        links = row.find_all("a", href=True)

        for link in links:
            href = link["href"]
            text = link.get_text(" ", strip=True)

            check = (href + " " + text).lower()

            # Skip non-data files
            bad_words = [
                "stata",
                "spss",
                "sas",
                "dict",
                "dictionary",
                "program"
            ]

            if any(bad in check for bad in bad_words):
                continue

            if ".zip" in check:
                return urljoin(page_url, href)

    return None


def choose_best_csv_info(zip_file):
    csv_infos = [
        info for info in zip_file.infolist()
        if info.filename.lower().endswith(".csv")
    ]

    if not csv_infos:
        return None

    rv_infos = [
        info for info in csv_infos
        if Path(info.filename).name.lower().endswith("_rv.csv")
    ]

    # Use revised file first if available
    if rv_infos:
        return max(rv_infos, key=lambda x: x.file_size)

    # Otherwise use normal CSV
    return max(csv_infos, key=lambda x: x.file_size)


def save_csv_year_filename_only(zip_file, csv_info, year):
    original_filename = Path(csv_info.filename).name.lower()

    # Example:
    # 2010_c2010_a_rv.csv
    output_filename = f"{year}_{original_filename}"
    output_path = folder / output_filename

    # Copy CSV directly.
    # Do NOT change inside the CSV.
    with zip_file.open(csv_info) as source, open(output_path, "wb") as output:
        shutil.copyfileobj(source, output)

    return output_filename


def download_and_save_one_year(year, data_file_url):
    zip_response = session.get(data_file_url, timeout=90)
    zip_response.raise_for_status()

    with zipfile.ZipFile(BytesIO(zip_response.content), "r") as zip_file:
        csv_info = choose_best_csv_info(zip_file)

        if csv_info is None:
            return None, "no csv found"

        saved_filename = save_csv_year_filename_only(
            zip_file=zip_file,
            csv_info=csv_info,
            year=year
        )

        if Path(csv_info.filename).name.lower().endswith("_rv.csv"):
            file_type = "revised"
        else:
            file_type = "normal"

        return saved_filename, file_type


# ==================================================
# MAIN AUTO DOWNLOAD
# ==================================================

total_start = time.perf_counter()

print("Auto download started")
print(f"Saving files into folder: {folder}")
print("-" * 60)

available_count = 0
not_available_count = 0
error_count = 0

for year in range(start_year, end_year + 1):
    year_start = time.perf_counter()

    print(f"Checking {year}...")

    try:
        page_url, html = get_year_page(year)

        data_file_url = find_data_file_link(html, page_url)

        if data_file_url is None:
            not_available_count += 1
            print("  Not available")
            print()
            continue

        saved_filename, file_type = download_and_save_one_year(
            year,
            data_file_url
        )

        year_seconds = round(time.perf_counter() - year_start, 2)
        total_so_far = round(time.perf_counter() - total_start, 2)

        if saved_filename is None:
            error_count += 1
            print(f"  Error: {file_type}")
            print()
            continue

        available_count += 1

        print(f"  Saved: {saved_filename}")
        print(f"  File type used: {file_type}")
        print(f"  Year time: {year_seconds} seconds")
        print(f"  Total time so far: {total_so_far} seconds")
        print()

    except Exception as e:
        error_count += 1

        year_seconds = round(time.perf_counter() - year_start, 2)

        print(f"  Error: {e}")
        print(f"  Year time: {year_seconds} seconds")
        print()


# ==================================================
# FINAL TIME
# ==================================================

total_seconds = round(time.perf_counter() - total_start, 2)
total_minutes = round(total_seconds / 60, 2)

print("-" * 60)
print("Auto download finished")
print(f"Total time: {total_seconds} seconds")
print(f"Total time: {total_minutes} minutes")
print(f"Available files saved: {available_count}")
print(f"Not available years: {not_available_count}")
print(f"Errors: {error_count}")
print(f"Folder: {folder}")

Auto download started
Saving files into folder: CorrectCheckYear
------------------------------------------------------------
Checking 2023...
  Not available

Checking 2024...
  Not available

------------------------------------------------------------
Auto download finished
Total time: 2.73 seconds
Total time: 0.05 minutes
Available files saved: 0
Not available years: 2
Errors: 0
Folder: CorrectCheckYear


In [30]:
########### Check why error - corrected full code ###########

import requests
from bs4 import BeautifulSoup
from pathlib import Path
from urllib.parse import urljoin
from io import BytesIO
import zipfile
import shutil
import time
import re

# ==================================================
# SETTINGS
# ==================================================

target_title = "Awards/degrees conferred by program (6-digit CIP code), award level"

start_year = 2020
end_year = 2024

folder = Path("CorrectCheckYear")

# True = delete old folder first
# This removes old wrong files before starting fresh
CLEAN_FOLDER_FIRST = True

base_url = "https://nces.ed.gov/ipeds/datacenter/DataFiles.aspx"

session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0"
})


# ==================================================
# CREATE CLEAN FOLDER
# ==================================================

if CLEAN_FOLDER_FIRST and folder.exists():
    shutil.rmtree(folder)

folder.mkdir(exist_ok=True)


# ==================================================
# HELPER FUNCTIONS
# ==================================================

def clean_text(text):
    return " ".join(text.lower().split())


def get_year_page(year):
    params = {
        "year": year,
        "rtid": 7,
        "surveyNumber": 3   # 3 = Completions survey
    }

    response = session.get(base_url, params=params, timeout=30)
    response.raise_for_status()

    return response.url, response.text


def find_data_file_link(html, page_url):
    soup = BeautifulSoup(html, "html.parser")
    target_clean = clean_text(target_title)

    rows = soup.find_all("tr")

    for row in rows:
        row_text = clean_text(row.get_text(" ", strip=True))

        # Find row with target title
        if target_clean not in row_text:
            continue

        cells = row.find_all(["td", "th"])

        # NCES table usually:
        # Year | Survey | Title | Data File | Stata Data File | Programs | Dictionary
        if len(cells) < 4:
            continue

        # Data File is usually the 4th column
        data_file_cell = cells[3]

        links = data_file_cell.find_all("a", href=True)

        for link in links:
            href = link["href"].strip()
            text = link.get_text(" ", strip=True).strip()

            check = (href + " " + text).lower()

            # Skip non-data-file links
            bad_words = [
                "stata",
                "spss",
                "sas",
                "dict",
                "dictionary",
                "program"
            ]

            if any(bad in check for bad in bad_words):
                continue

            # Case 1: href already has .zip
            if ".zip" in href.lower():
                return urljoin(page_url, href)

            # Case 2: link text is like C2023_A
            # Build the zip URL manually
            if re.fullmatch(r"[A-Za-z0-9_]+", text):
                return urljoin(page_url, f"data/{text}.zip")

    return None


def choose_best_csv_info(zip_file):
    csv_infos = [
        info for info in zip_file.infolist()
        if info.filename.lower().endswith(".csv")
    ]

    if not csv_infos:
        return None

    rv_infos = [
        info for info in csv_infos
        if Path(info.filename).name.lower().endswith("_rv.csv")
    ]

    # Use revised file first if available
    if rv_infos:
        return max(rv_infos, key=lambda x: x.file_size)

    # Otherwise use normal CSV
    return max(csv_infos, key=lambda x: x.file_size)


def save_csv_year_filename_only(zip_file, csv_info, year):
    original_filename = Path(csv_info.filename).name.lower()

    # Example output:
    # 2023_c2023_a.csv
    output_filename = f"{year}_{original_filename}"
    output_path = folder / output_filename

    # Copy CSV directly.
    # This does NOT change anything inside the CSV.
    with zip_file.open(csv_info) as source, open(output_path, "wb") as output:
        shutil.copyfileobj(source, output)

    return output_filename


def download_and_save_one_year(year, data_file_url):
    zip_response = session.get(data_file_url, timeout=90)
    zip_response.raise_for_status()

    # Safety check:
    # Real zip files start with bytes PK.
    # If not, NCES probably returned an HTML error page.
    if not zip_response.content.startswith(b"PK"):
        return None, f"not a zip file: {data_file_url}"

    with zipfile.ZipFile(BytesIO(zip_response.content), "r") as zip_file:
        csv_info = choose_best_csv_info(zip_file)

        if csv_info is None:
            return None, "no csv found"

        saved_filename = save_csv_year_filename_only(
            zip_file=zip_file,
            csv_info=csv_info,
            year=year
        )

        if Path(csv_info.filename).name.lower().endswith("_rv.csv"):
            file_type = "revised"
        else:
            file_type = "normal"

        return saved_filename, file_type


# ==================================================
# MAIN AUTO DOWNLOAD
# ==================================================

total_start = time.perf_counter()

print("Auto download started")
print(f"Saving files into folder: {folder}")
print("-" * 60)

available_count = 0
not_available_count = 0
error_count = 0

for year in range(start_year, end_year + 1):
    year_start = time.perf_counter()

    print(f"Checking {year}...")

    try:
        page_url, html = get_year_page(year)

        data_file_url = find_data_file_link(html, page_url)

        if data_file_url is None:
            not_available_count += 1
            print("  Not available")
            print()
            continue

        print(f"  Download URL: {data_file_url}")

        saved_filename, file_type = download_and_save_one_year(
            year,
            data_file_url
        )

        year_seconds = round(time.perf_counter() - year_start, 2)
        total_so_far = round(time.perf_counter() - total_start, 2)

        if saved_filename is None:
            error_count += 1
            print(f"  Error: {file_type}")
            print(f"  Year time: {year_seconds} seconds")
            print()
            continue

        available_count += 1

        print(f"  Saved: {saved_filename}")
        print(f"  File type used: {file_type}")
        print(f"  Year time: {year_seconds} seconds")
        print(f"  Total time so far: {total_so_far} seconds")
        print()

    except Exception as e:
        error_count += 1

        year_seconds = round(time.perf_counter() - year_start, 2)

        print(f"  Error: {e}")
        print(f"  Year time: {year_seconds} seconds")
        print()


# ==================================================
# FINAL TIME
# ==================================================

total_seconds = round(time.perf_counter() - total_start, 2)
total_minutes = round(total_seconds / 60, 2)

print("-" * 60)
print("Auto download finished")
print(f"Total time: {total_seconds} seconds")
print(f"Total time: {total_minutes} minutes")
print(f"Available files saved: {available_count}")
print(f"Not available years: {not_available_count}")
print(f"Errors: {error_count}")
print(f"Folder: {folder}")

Auto download started
Saving files into folder: CorrectCheckYear
------------------------------------------------------------
Checking 2020...
  Download URL: https://nces.ed.gov/ipeds/datacenter/data/C2020_A.zip
  Saved: 2020_c2020_a_rv.csv
  File type used: revised
  Year time: 23.77 seconds
  Total time so far: 23.77 seconds

Checking 2021...
  Download URL: https://nces.ed.gov/ipeds/datacenter/data/C2021_A.zip
  Saved: 2021_c2021_a_rv.csv
  File type used: revised
  Year time: 41.9 seconds
  Total time so far: 65.67 seconds

Checking 2022...
  Download URL: https://nces.ed.gov/ipeds/datacenter/data/C2022_A.zip
  Saved: 2022_c2022_a_rv.csv
  File type used: revised
  Year time: 35.84 seconds
  Total time so far: 101.51 seconds

Checking 2023...
  Download URL: https://nces.ed.gov/ipeds/datacenter/data/C2023_A.zip
  Saved: 2023_c2023_a_rv.csv
  File type used: revised
  Year time: 30.54 seconds
  Total time so far: 132.05 seconds

Checking 2024...
  Download URL: https://nces.ed.gov/

# Load

In [33]:
########### Corrected full code with overall loading bar ###########

import requests
from bs4 import BeautifulSoup
from pathlib import Path
from urllib.parse import urljoin
from io import BytesIO
import zipfile
import shutil
import time
import re
import sys

# ==================================================
# SETTINGS
# ==================================================

target_title = "Awards/degrees conferred by program (6-digit CIP code), award level"

start_year = 2023
end_year = 2024

folder = Path("CorrectCheckYear")

# True = delete old folder first
CLEAN_FOLDER_FIRST = True

base_url = "https://nces.ed.gov/ipeds/datacenter/DataFiles.aspx"

session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0"
})


# ==================================================
# CREATE CLEAN FOLDER
# ==================================================

if CLEAN_FOLDER_FIRST and folder.exists():
    shutil.rmtree(folder)

folder.mkdir(exist_ok=True)


# ==================================================
# HELPER FUNCTIONS
# ==================================================

def clean_text(text):
    return " ".join(text.lower().split())


def show_progress(current, total, year):
    percent = current / total
    bar_length = 30
    filled_length = int(bar_length * percent)

    bar = "█" * filled_length + "-" * (bar_length - filled_length)

    message = f"\rLoading all years: |{bar}| {current}/{total} ({percent:.1%}) Current year: {year}"
    sys.stdout.write(message)
    sys.stdout.flush()


def get_year_page(year):
    params = {
        "year": year,
        "rtid": 7,
        "surveyNumber": 3
    }

    response = session.get(base_url, params=params, timeout=30)
    response.raise_for_status()

    return response.url, response.text


def find_data_file_link(html, page_url):
    soup = BeautifulSoup(html, "html.parser")
    target_clean = clean_text(target_title)

    rows = soup.find_all("tr")

    for row in rows:
        row_text = clean_text(row.get_text(" ", strip=True))

        if target_clean not in row_text:
            continue

        cells = row.find_all(["td", "th"])

        if len(cells) < 4:
            continue

        data_file_cell = cells[3]
        links = data_file_cell.find_all("a", href=True)

        for link in links:
            href = link["href"].strip()
            text = link.get_text(" ", strip=True).strip()

            check = (href + " " + text).lower()

            bad_words = [
                "stata",
                "spss",
                "sas",
                "dict",
                "dictionary",
                "program"
            ]

            if any(bad in check for bad in bad_words):
                continue

            if ".zip" in href.lower():
                return urljoin(page_url, href)

            if re.fullmatch(r"[A-Za-z0-9_]+", text):
                return urljoin(page_url, f"data/{text}.zip")

    return None


def choose_best_csv_info(zip_file):
    csv_infos = [
        info for info in zip_file.infolist()
        if info.filename.lower().endswith(".csv")
    ]

    if not csv_infos:
        return None

    rv_infos = [
        info for info in csv_infos
        if Path(info.filename).name.lower().endswith("_rv.csv")
    ]

    if rv_infos:
        return max(rv_infos, key=lambda x: x.file_size)

    return max(csv_infos, key=lambda x: x.file_size)


def save_csv_year_filename_only(zip_file, csv_info, year):
    original_filename = Path(csv_info.filename).name.lower()

    output_filename = f"{year}_{original_filename}"
    output_path = folder / output_filename

    # Copy CSV directly.
    # This does NOT change anything inside the CSV.
    with zip_file.open(csv_info) as source, open(output_path, "wb") as output:
        shutil.copyfileobj(source, output)

    return output_filename


def download_and_save_one_year(year, data_file_url):
    zip_response = session.get(data_file_url, timeout=90)
    zip_response.raise_for_status()

    if not zip_response.content.startswith(b"PK"):
        return None, f"not a zip file: {data_file_url}"

    with zipfile.ZipFile(BytesIO(zip_response.content), "r") as zip_file:
        csv_info = choose_best_csv_info(zip_file)

        if csv_info is None:
            return None, "no csv found"

        saved_filename = save_csv_year_filename_only(
            zip_file=zip_file,
            csv_info=csv_info,
            year=year
        )

        if Path(csv_info.filename).name.lower().endswith("_rv.csv"):
            file_type = "revised"
        else:
            file_type = "normal"

        return saved_filename, file_type


# ==================================================
# MAIN AUTO DOWNLOAD WITH ONE OVERALL LOADING BAR
# ==================================================

total_start = time.perf_counter()

years = list(range(start_year, end_year + 1))
total_years = len(years)

available_count = 0
not_available_count = 0
error_count = 0

saved_files = []
not_available_years = []
error_years = []

print("Auto download started")
print(f"Saving files into folder: {folder}")
print("-" * 60)

for index, year in enumerate(years, start=1):
    show_progress(index, total_years, year)

    try:
        page_url, html = get_year_page(year)

        data_file_url = find_data_file_link(html, page_url)

        if data_file_url is None:
            not_available_count += 1
            not_available_years.append(year)
            continue

        saved_filename, file_type = download_and_save_one_year(
            year,
            data_file_url
        )

        if saved_filename is None:
            error_count += 1
            error_years.append((year, file_type))
            continue

        available_count += 1
        saved_files.append((year, saved_filename, file_type))

    except Exception as e:
        error_count += 1
        error_years.append((year, str(e)))


# Move to new line after loading bar finishes
print()
print("-" * 60)


# ==================================================
# FINAL SUMMARY ONLY
# ==================================================

total_seconds = round(time.perf_counter() - total_start, 2)
total_minutes = round(total_seconds / 60, 2)

print("Auto download finished")
print(f"Total time: {total_seconds} seconds")
print(f"Total time: {total_minutes} minutes")
print(f"Available files saved: {available_count}")
print(f"Not available years: {not_available_count}")
print(f"Errors: {error_count}")
print(f"Folder: {folder}")

print()
print("Saved files:")
for year, filename, file_type in saved_files:
    print(f"  {year}: {filename} ({file_type})")

if not_available_years:
    print()
    print("Not available years:")
    print(" ", not_available_years)

if error_years:
    print()
    print("Error years:")
    for year, error in error_years:
        print(f"  {year}: {error}")

Auto download started
Saving files into folder: CorrectCheckYear
------------------------------------------------------------
Loading all years: |██████████████████████████████| 2/2 (100.0%) Current year: 2024
------------------------------------------------------------
Auto download finished
Total time: 16.82 seconds
Total time: 0.28 minutes
Available files saved: 2
Not available years: 0
Errors: 0
Folder: CorrectCheckYear

Saved files:
  2023: 2023_c2023_a_rv.csv (revised)
  2024: 2024_c2024_a.csv (normal)


# Load with time

In [36]:
########### Corrected full code with overall loading bar + time left ###########

import requests
from bs4 import BeautifulSoup
from pathlib import Path
from urllib.parse import urljoin
from io import BytesIO
import zipfile
import shutil
import time
import re
import sys

# ==================================================
# SETTINGS
# ==================================================

target_title = "Awards/degrees conferred by program (6-digit CIP code), award level"

start_year = 2023
end_year = 2024

folder = Path("CorrectCheckYear")

# True = delete old folder first
CLEAN_FOLDER_FIRST = True

base_url = "https://nces.ed.gov/ipeds/datacenter/DataFiles.aspx"

session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0"
})


# ==================================================
# CREATE CLEAN FOLDER
# ==================================================

if CLEAN_FOLDER_FIRST and folder.exists():
    shutil.rmtree(folder)

folder.mkdir(exist_ok=True)


# ==================================================
# HELPER FUNCTIONS
# ==================================================

def clean_text(text):
    return " ".join(text.lower().split())


def format_seconds(seconds):
    seconds = int(seconds)

    minutes = seconds // 60
    seconds = seconds % 60

    if minutes >= 60:
        hours = minutes // 60
        minutes = minutes % 60
        return f"{hours}h {minutes}m {seconds}s"

    if minutes > 0:
        return f"{minutes}m {seconds}s"

    return f"{seconds}s"


def show_progress(done, total, year, start_time):
    percent = done / total
    bar_length = 30
    filled_length = int(bar_length * percent)

    bar = "█" * filled_length + "-" * (bar_length - filled_length)

    elapsed = time.perf_counter() - start_time
    elapsed_text = format_seconds(elapsed)

    if done == 0:
        eta_text = "calculating..."
    else:
        avg_time_per_year = elapsed / done
        remaining_years = total - done
        eta_seconds = avg_time_per_year * remaining_years
        eta_text = format_seconds(eta_seconds)

    message = (
        f"\rLoading all years: |{bar}| "
        f"{done}/{total} ({percent:.1%}) "
        f"Current year: {year} "
        f"Elapsed: {elapsed_text} "
        f"Time left: {eta_text}   "
    )

    sys.stdout.write(message)
    sys.stdout.flush()


def get_year_page(year):
    params = {
        "year": year,
        "rtid": 7,
        "surveyNumber": 3
    }

    response = session.get(base_url, params=params, timeout=30)
    response.raise_for_status()

    return response.url, response.text


def find_data_file_link(html, page_url):
    soup = BeautifulSoup(html, "html.parser")
    target_clean = clean_text(target_title)

    rows = soup.find_all("tr")

    for row in rows:
        row_text = clean_text(row.get_text(" ", strip=True))

        if target_clean not in row_text:
            continue

        cells = row.find_all(["td", "th"])

        # NCES table usually:
        # Year | Survey | Title | Data File | Stata Data File | Programs | Dictionary
        if len(cells) < 4:
            continue

        # Data File is usually the 4th column
        data_file_cell = cells[3]
        links = data_file_cell.find_all("a", href=True)

        for link in links:
            href = link["href"].strip()
            text = link.get_text(" ", strip=True).strip()

            check = (href + " " + text).lower()

            bad_words = [
                "stata",
                "spss",
                "sas",
                "dict",
                "dictionary",
                "program"
            ]

            if any(bad in check for bad in bad_words):
                continue

            # Case 1: href already has .zip
            if ".zip" in href.lower():
                return urljoin(page_url, href)

            # Case 2: link text is like C2023_A
            # Build zip URL manually
            if re.fullmatch(r"[A-Za-z0-9_]+", text):
                return urljoin(page_url, f"data/{text}.zip")

    return None


def choose_best_csv_info(zip_file):
    csv_infos = [
        info for info in zip_file.infolist()
        if info.filename.lower().endswith(".csv")
    ]

    if not csv_infos:
        return None

    rv_infos = [
        info for info in csv_infos
        if Path(info.filename).name.lower().endswith("_rv.csv")
    ]

    # Use revised file first if available
    if rv_infos:
        return max(rv_infos, key=lambda x: x.file_size)

    # Otherwise use normal CSV
    return max(csv_infos, key=lambda x: x.file_size)


def save_csv_year_filename_only(zip_file, csv_info, year):
    original_filename = Path(csv_info.filename).name.lower()

    # Example output:
    # 2023_c2023_a.csv
    output_filename = f"{year}_{original_filename}"
    output_path = folder / output_filename

    # Copy CSV directly.
    # This does NOT change anything inside the CSV.
    with zip_file.open(csv_info) as source, open(output_path, "wb") as output:
        shutil.copyfileobj(source, output)

    return output_filename


def download_and_save_one_year(year, data_file_url):
    zip_response = session.get(data_file_url, timeout=90)
    zip_response.raise_for_status()

    # Safety check:
    # Real zip files start with bytes PK.
    # If not, NCES probably returned an HTML error page.
    if not zip_response.content.startswith(b"PK"):
        return None, f"not a zip file: {data_file_url}"

    with zipfile.ZipFile(BytesIO(zip_response.content), "r") as zip_file:
        csv_info = choose_best_csv_info(zip_file)

        if csv_info is None:
            return None, "no csv found"

        saved_filename = save_csv_year_filename_only(
            zip_file=zip_file,
            csv_info=csv_info,
            year=year
        )

        if Path(csv_info.filename).name.lower().endswith("_rv.csv"):
            file_type = "revised"
        else:
            file_type = "normal"

        return saved_filename, file_type


# ==================================================
# MAIN AUTO DOWNLOAD WITH ONE OVERALL LOADING BAR
# ==================================================

total_start = time.perf_counter()

years = list(range(start_year, end_year + 1))
total_years = len(years)

available_count = 0
not_available_count = 0
error_count = 0

saved_files = []
not_available_years = []
error_years = []

print("Auto download started")
print(f"Saving files into folder: {folder}")
print("-" * 60)

for index, year in enumerate(years, start=1):

    # Show progress before starting current year
    show_progress(index - 1, total_years, year, total_start)

    try:
        page_url, html = get_year_page(year)

        data_file_url = find_data_file_link(html, page_url)

        if data_file_url is None:
            not_available_count += 1
            not_available_years.append(year)

            # Update progress after finishing current year
            show_progress(index, total_years, year, total_start)
            continue

        saved_filename, file_type = download_and_save_one_year(
            year,
            data_file_url
        )

        if saved_filename is None:
            error_count += 1
            error_years.append((year, file_type))

            # Update progress after finishing current year
            show_progress(index, total_years, year, total_start)
            continue

        available_count += 1
        saved_files.append((year, saved_filename, file_type))

        # Update progress after finishing current year
        show_progress(index, total_years, year, total_start)

    except Exception as e:
        error_count += 1
        error_years.append((year, str(e)))

        # Update progress after finishing current year
        show_progress(index, total_years, year, total_start)


# Move to new line after loading bar finishes
print()
print("-" * 60)


# ==================================================
# FINAL SUMMARY ONLY
# ==================================================

total_seconds = round(time.perf_counter() - total_start, 2)
total_minutes = round(total_seconds / 60, 2)

print("Auto download finished")
print(f"Total time: {total_seconds} seconds")
print(f"Total time: {total_minutes} minutes")
print(f"Available files saved: {available_count}")
print(f"Not available years: {not_available_count}")
print(f"Errors: {error_count}")
print(f"Folder: {folder}")

print()
print("Saved files:")
for year, filename, file_type in saved_files:
    print(f"  {year}: {filename} ({file_type})")

if not_available_years:
    print()
    print("Not available years:")
    print(" ", not_available_years)

if error_years:
    print()
    print("Error years:")
    for year, error in error_years:
        print(f"  {year}: {error}")

Auto download started
Saving files into folder: CorrectCheckYear
------------------------------------------------------------
Loading all years: |██████████████████████████████| 2/2 (100.0%) Current year: 2024 Elapsed: 49s Time left: 0s   ing...   
------------------------------------------------------------
Auto download finished
Total time: 49.28 seconds
Total time: 0.82 minutes
Available files saved: 2
Not available years: 0
Errors: 0
Folder: CorrectCheckYear

Saved files:
  2023: 2023_c2023_a_rv.csv (revised)
  2024: 2024_c2024_a.csv (normal)
